# 04 · Comparing the three counting methods
**Goal:** do kallisto, STAR and RSEM give the same biological answers?
**Input:** kallisto, STAR and RSEM results from Euler (notebook 01). **Output:** summary tables in `results/compare/`.
**Kernel:** **R** (`ir`) · **Time:** ~5 min.

| Method | Reads go to | Counts |
|---|---|---|
| **kallisto** | transcripts (pseudoalignment); EM shares reads between isoforms | estimated counts |
| **STAR** (`GeneCounts`) | the genome; only uniquely mapped reads are counted per gene | whole numbers |
| **RSEM** | STAR's alignments to transcripts (a counting step after STAR, not an independent mapping); EM shares multi-mapping reads | expected counts |

All three start from the same trimmed reads and the same GENCODE M39 annotation, and go through the same DESeq2 code (`run_dds`, `run_de`), including the exclusion of outlier SRR21355389 from the aging analysis (notebook 03). Differences between STAR and RSEM come from the counting step only (same alignments); kallisto differs in mapping as well. Measures used: **Spearman correlation** (agreement of rankings; 1 = identical), **Jaccard index** (shared ÷ combined number of DEGs), and *same direction* (share of shared DEGs whose log2FC has the same sign). See [docs/glossary.md](../docs/glossary.md).

In [ ]:
# txi: kallisto counts; rsem: RSEM counts (both tximport lists); star: STAR counts (genes x runs); all use GENCODE M39 gene ids
source('../R/helpers.R')
samples <- load_samples()
x <- load_txi(rownames(samples)); txi <- x$txi; genes <- x$genes
rsem <- load_rsem(rownames(samples))
star <- load_star_counts()[, rownames(samples)]
ids <- Reduce(intersect, list(rownames(txi$counts), rownames(rsem$counts), rownames(star)))
cat('genes: kallisto', nrow(txi$counts), '| RSEM', nrow(rsem$counts), '| STAR', nrow(star), '| shared by all three', length(ids), '\n')

## 1 · Counts per sample
Per sample, Spearman correlation of gene counts between each pair of methods, and the total counts of each method (millions). STAR counts only uniquely mapped reads, so its totals are lower; kallisto and RSEM also count reads that fit several places.

In [ ]:
# one row per sample: correlation per pair of methods and total counts (millions) per method, over genes present in all three
m <- list(kallisto = txi$counts[ids, ], STAR = star[ids, ], RSEM = rsem$counts[ids, ])
rho <- \(a, b) map_dbl(rownames(samples), \(s) cor(m[[a]][, s], m[[b]][, s], method = 'spearman'))
per_sample <- tibble(srr = rownames(samples), group = samples$group,
  spearman_kallisto_STAR = rho('kallisto', 'STAR'), spearman_kallisto_RSEM = rho('kallisto', 'RSEM'), spearman_STAR_RSEM = rho('STAR', 'RSEM'),
  kallisto_M = colSums(m$kallisto) / 1e6, STAR_M = colSums(m$STAR) / 1e6, RSEM_M = colSums(m$RSEM) / 1e6)
per_sample

## 2 · Differential expression with each method

In [ ]:
# the four DESeq2 analyses of notebook 03 for each method (dds = fitted models, de = result tables); counts of up / down DEGs
counts <- list(kallisto = txi, STAR = star, RSEM = rsem)
dds <- lapply(counts, run_dds, samples = samples)
de <- lapply(dds, de_tables, genes = genes)
bind_rows(lapply(de, \(r) bind_rows(lapply(r, n_deg), .id = 'contrast')), .id = 'method')

### Agreement per contrast
`method_a` and `method_b` are the two methods compared. `total_DEG_a` / `_b` = all DEGs of each method (as in the table above). The other columns use only genes with a valid p-value in **both** methods (`genes_tested_both`; each method filters low-count genes itself), so `DEG_a` / `DEG_b` can be smaller than the totals.
`jaccard` = shared ÷ combined DEGs; `same_direction` = share of shared DEGs with the same sign; `lfc_spearman_all` / `_sig` = rank correlation of log2FC over all tested genes / over genes significant in either method.

In [ ]:
# agreement table: every pair of methods, every comparison
pairs <- list(c('kallisto', 'STAR'), c('kallisto', 'RSEM'), c('STAR', 'RSEM'))
cmp <- bind_rows(lapply(pairs, \(p) bind_rows(lapply(names(de$kallisto), \(n)
  compare_contrast(de[[p[1]]][[n]], de[[p[2]]][[n]]) %>% mutate(contrast = n))) %>% mutate(method_a = p[1], method_b = p[2]))) %>%
  relocate(method_a, method_b, contrast)
cmp

In [ ]:
# log2FC of one method (x) against another (y) per gene; colour = which method calls the gene a DEG. Change PLOT_PAIR to see another pair.
PLOT_PAIR <- c('kallisto', 'RSEM')
for (n in names(de$kallisto)) print(lfc_scatter(de[[PLOT_PAIR[1]]][[n]], de[[PLOT_PAIR[2]]][[n]], n, names = PLOT_PAIR))

## 3 · Genes called by only one method
Are the disagreements mostly lowly expressed genes? `median_baseMean` = median of the genes' average normalised count.

In [ ]:
# per pair and comparison: genes called by both / only by the first / only by the second method, with typical expression
only <- function(a, b, alpha = 0.05) inner_join(a, b, by = 'gene_id', suffix = c('_a', '_b')) %>% filter(!is.na(padj_a), !is.na(padj_b)) %>%
  mutate(call = case_when(padj_a < alpha & padj_b < alpha ~ 'both', padj_a < alpha ~ 'first only', padj_b < alpha ~ 'second only', TRUE ~ 'neither'))
bind_rows(lapply(pairs, \(p) bind_rows(lapply(names(de$kallisto), \(cn) only(de[[p[1]]][[cn]], de[[p[2]]][[cn]]) %>% filter(call != 'neither') %>%
  group_by(call) %>% summarise(n_genes = n(), median_baseMean = round(median((baseMean_a + baseMean_b) / 2)), .groups = 'drop') %>% mutate(contrast = cn))) %>%
  mutate(first = p[1], second = p[2]))) %>% relocate(first, second, contrast)

## 4 · Genes highlighted in the paper
Aging genes (C4b, Trem2, …) and circadian-clock genes induced by dietary restriction (Cry1, Usp2, Ciart, …) with each method.

In [ ]:
# log2FC and padj of the paper's genes, the three methods side by side
key <- list(aging_21v3 = c('C4b', 'Trem2', 'Gpr17', 'Gfap', 'Ctss', 'Tyrobp'), aDR = c('Cry1', 'Usp2', 'Ciart', 'Cirbp', 'Dbp'))
bind_rows(lapply(names(key), \(n) {
  f <- \(m) de[[m]][[n]] %>% filter(symbol %in% key[[n]]) %>% transmute(contrast = n, symbol, !!paste0('lfc_', m) := round(log2FoldChange, 2), !!paste0('padj_', m) := signif(padj, 2))
  purrr::reduce(lapply(names(de), f), inner_join, by = c('contrast', 'symbol'))
}))

## 6 · Evidence for choosing a method
No method is assumed to be the best. The measurements below are fixed in advance (see "Choosing the counting method" in [docs/pipeline.md](../docs/pipeline.md)); each one says something about data use, noise or stability, and none uses the biological conclusion. Read them together: a method is a poor choice only if it is clearly worse on several of them. The positive-control rows are a sanity check, not a selection criterion (using them to pick a method would be circular).

### 6a · Usable data and replicate agreement

In [ ]:
# usable data: share of the trimmed read pairs that end up counted in a gene; genes detected (>= 10 counts in >= 3 runs)
inp <- read_csv(file.path(ROOT, 'results/star/star_qc.csv'), show_col_types = FALSE) %>% column_to_rownames('SRR')
pairs_in <- inp[rownames(samples), 'input_pairs_M'] * 1e6          # pairs that entered kallisto and STAR
cnt <- lapply(counts, \(m) if (is.matrix(m)) m[, rownames(samples)] else m$counts[, rownames(samples)])
usable <- bind_rows(lapply(names(cnt), \(m) tibble(method = m, counted_pairs_pct = 100 * colSums(cnt[[m]]) / pairs_in)))
use_tab <- usable %>% group_by(method) %>% summarise(counted_pairs_pct_median = round(median(counted_pairs_pct), 1), min = round(min(counted_pairs_pct), 1), max = round(max(counted_pairs_pct), 1)) %>%
  left_join(tibble(method = names(cnt), genes_detected = map_int(cnt, \(m) sum(rowSums(m >= 10) >= 3))), by = 'method')
use_tab

In [ ]:
# replicate agreement: median pairwise Pearson r of log2 CPM between the mice of a group (genes present and expressed in all three methods)
keep <- ids[Reduce(`&`, lapply(cnt, \(m) rowMeans(m[ids, ]) >= 10))]
lcpm <- lapply(cnt, \(m) { m <- m[keep, ]; log2(sweep(m, 2, colSums(m) / 1e6, '/') + 1) })
rep_tab <- bind_rows(lapply(names(lcpm), \(mth) bind_rows(lapply(split(rownames(samples), samples$group), \(s) {
  cm <- cor(lcpm[[mth]][, s]); tibble(n_mice = length(s), median_pairwise_r = median(cm[lower.tri(cm)])) }), .id = 'group') %>% mutate(method = mth))) %>% relocate(method)
rep_tab %>% mutate(median_pairwise_r = round(median_pairwise_r, 4)) %>% pivot_wider(names_from = method, values_from = median_pairwise_r)

### 6b · Noise and stability
`dispersion` = median gene-wise DESeq2 dispersion over the genes tested by all three methods (unexplained variation between mice of a group; lower = more power, but compare with care because the count scales differ). `DEG_without` / `DEG_with` = aging DEGs (21 vs 3 months) without and with the outlier SRR21355389, `jaccard_outlier` = their overlap (1 = the call does not depend on that mouse).

In [ ]:
# dispersion per model and stability of the aging DEGs to the outlier
tested <- Reduce(intersect, lapply(dds, \(d) Reduce(intersect, lapply(d, rownames))))
disp <- bind_rows(lapply(names(dds), \(m) tibble(method = m, !!!lapply(dds[[m]], \(d) round(median(setNames(dispersions(d), rownames(d))[intersect(rownames(d), tested)], na.rm = TRUE), 3)))))
s_all <- samples %>% filter(cohort == 'aging') %>% mutate(group = droplevels(group))
stab <- bind_rows(lapply(names(counts), \(m) {
  all12 <- de_table(make_dds(counts[[m]], s_all, ~ sex + group, min_n = 4), c('group', 'aging 21mo', 'aging 3mo'), genes)
  cd <- compare_de(de[[m]]$aging_21v3, all12)
  tibble(method = m, DEG_without = cd$DEG_a, DEG_with = cd$DEG_b, jaccard_outlier = round(cd$DEG_both / (cd$DEG_a + cd$DEG_b - cd$DEG_both), 2)) }))
left_join(rename_with(disp, ~ paste0('dispersion_', .x), -method), stab, by = 'method')

### 6c · Positive controls and agreement with the other methods

In [ ]:
# positive controls: genes the paper reports as up with aging (5 inflammation genes) and as up with aDR (23 circadian/cycling genes); mean Jaccard with the other two methods
up_aging <- c('C4b', 'Trem2', 'Ctss', 'Tyrobp', 'Gfap')
adr23 <- c('Chordc1','Cxcl12','Fabp7','Kdr','Adipor2','Gjb6','Mertk','Cry1','Klf15','Dbp','Usp2','Etnppl','Ciart','Sult1a1','Zbtb16','Fkbp5','Sgk1','Tsc22d3','Cirbp','Rbm3','Plin4','Fmo2','Hif3a')
ctrl <- bind_rows(lapply(names(de), \(m) tibble(method = m,
  aging_genes_up_of_5 = sum(filter(de[[m]]$aging_21v3, symbol %in% up_aging)$log2FoldChange > 0),
  aDR_genes_up_of_23 = sum(filter(de[[m]]$aDR, symbol %in% adr23)$log2FoldChange > 0),
  mean_jaccard_with_others = round(mean(filter(cmp, method_a == m | method_b == m)$jaccard), 2))))
ctrl

### 6d · The evidence table
One row per method, saved to `results/compare/method_evidence.csv`. Practical criteria that are not computed here (run time and memory on Euler, software and reference requirements) are listed in [docs/pipeline.md](../docs/pipeline.md).

In [ ]:
# all measurements together, one row per method
evidence <- use_tab %>% left_join(rep_tab %>% group_by(method) %>% summarise(replicate_r_median = round(median(median_pairwise_r), 4)), by = 'method') %>%
  left_join(rename_with(disp, ~ paste0('dispersion_', .x), -method), by = 'method') %>% left_join(stab, by = 'method') %>% left_join(ctrl, by = 'method')
evidence

## 7 · Save

In [ ]:
# agreement tables for the report (per-gene tables of STAR and RSEM stay out of git)
dir.create(file.path(ROOT, 'results/compare'), showWarnings = FALSE)
write_csv(cmp, file.path(ROOT, 'results/compare/de_agreement.csv'))
write_csv(per_sample, file.path(ROOT, 'results/compare/counts_per_sample.csv'))
write_csv(evidence, file.path(ROOT, 'results/compare/method_evidence.csv'))
for (m in c('STAR', 'RSEM')) imap(de[[m]], \(r, n) write_csv(r, file.path(ROOT, 'results/compare', paste0(tolower(m), '_', n, '.csv')))) %>% invisible()
list.files(file.path(ROOT, 'results/compare'))